# Credit Risk Modeling (FinTech Case Study)

## Objective
Build a machine learning model to predict whether a borrower will default within 2 years.

Dataset: Give Me Some Credit (Kaggle)

Target Variable:
- `SeriousDlqin2yrs`
    - 1 = Default
    - 0 = No default

Business Context:
Banks want to minimize financial loss by:
- Catching defaulters
- Avoiding rejecting good customers

In [ ]:
import pandas as pd
# load data
df = pd.read_csv("data/cs-training.csv")
# drop useless index column
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df.head()


In [ ]:
# Dataset shape (To understand the data )
df.shape

In [ ]:
# check if numeric or categorical -  check missing values
df.info()

In [ ]:
# Target distribution (shows default or non defaults)
df["SeriousDlqin2yrs"].value_counts(normalize=True)

In [ ]:
# check missing values 
df.isna().sum()

In [ ]:
# create flag for missing income as sometimes the missing income itself signals risk. 
df["MonthlyIncome_missing"]=df["MonthlyIncome"].isna().astype(int)
df.groupby("MonthlyIncome_missing") ["SeriousDlqin2yrs"].mean()

In [ ]:
df["MonthlyIncome"].isna().mean()

In [ ]:
median_income=df["MonthlyIncome"].mean()
df["MonthlyIncome"]=df["MonthlyIncome"].fillna(median_income)

In [ ]:
df["MonthlyIncome"].isna().sum()

In [ ]:
df.corr(numeric_only=True)["SeriousDlqin2yrs"].sort_values(ascending=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.boxplot(x="SeriousDlqin2yrs", y="NumberOfTimes90DaysLate",data=df)
plt.title("Over 90 Days late vs Default")
plt.show()

In [ ]:
df.groupby("SeriousDlqin2yrs")["NumberOfTimes90DaysLate"].describe()

In [ ]:
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

for c in late_cols:
    df[c] = df[c].clip(lower=0, upper=10)

# Revolving utilization should typically be 0-1, but can exceed 1; cap at 1.5 for stability
df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(lower=0, upper=1.5)

# DebtRatio can be extreme; cap at 5 (you can justify as "extreme leverage/outliers")
df["DebtRatio"] = df["DebtRatio"].clip(lower=0, upper=5)

df[late_cols + ["RevolvingUtilizationOfUnsecuredLines","DebtRatio"]].describe().T[["min","50%","max"]]


In [ ]:
# train test split- essential for imbalanced data set 
from sklearn.model_selection import train_test_split
target="SeriousDlqin2yrs"
X=df.drop(columns=[target])
y=df[target]
X_train, X_test, y_train, y_test=train_test_split(
X,y, test_size=0.2 , random_state=42 , stratify=y
)

X_train.shape, X_test.shape

In [ ]:
# Build Logistic Regression Pipeline (ensure consistant pre-processing)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

log_reg = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_reg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
# Predicted probabilities
proba_test = log_reg.predict_proba(X_test)[:, 1]
# ROC-AUC
roc = roc_auc_score(y_test, proba_test)
roc

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report
# PR-AUC
pr_auc = average_precision_score(y_test, proba_test)
# Confusion Matrix
pred_05 = (proba_test >= 0.5).astype(int)

print("ROC-AUC:", roc)
print("PR-AUC:", pr_auc)
print("Confusion matrix @0.5:\n", confusion_matrix(y_test, pred_05))
print("\nReport @0.5:\n", classification_report(y_test, pred_05, digits=3))

## Interpretation

- ROC-AUC ≈ 0.86
- Model has strong ranking ability
- Recall for defaulters ≈ 75%
- Precision low due to class imbalance

Accuracy is misleading due to imbalance.

Next step:
Threshold optimisation using financial cost.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

RocCurveDisplay.from_predictions(y_test, proba_test)
plt.title("ROC Curve — Logistic Regression (Baseline)")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, proba_test)
plt.title("Precision–Recall Curve — Logistic Regression (Baseline)")
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

COST_FN = 10000  # approved defaulter (missed default)
COST_FP = 1000   # rejected good borrower (lost profit)

thresholds = np.linspace(0.01, 0.99, 99)
costs = []

for t in thresholds:
    preds = (proba_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    costs.append(COST_FN * fn + COST_FP * fp)

best_i = int(np.argmin(costs))
best_t = thresholds[best_i]

print("Best threshold:", best_t)
print("Min expected cost:", costs[best_i])

best_preds = (proba_test >= best_t).astype(int)
print("Confusion matrix @best_t:\n", confusion_matrix(y_test, best_preds))
print("\nReport @best_t:\n", classification_report(y_test, best_preds, digits=3))

plt.plot(thresholds, costs)
plt.title("Expected Cost vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Expected Cost")
plt.show()

# Gradient Boosting Model (HistGradientBoosting)

## Why Gradient Boosting?

Logistic regression assumes:
- Linear relationship between features and log-odds
- Additive effects

However, credit risk often includes:
- Non-linear relationships
- Interactions between variables
- Threshold effects

Gradient Boosting can capture:
- Non-linear patterns
- Feature interactions
- Complex decision boundaries


In [ ]:
# HIST GRADIENT BOOSTING MODEL
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(
    random_state=42
)

hgb.fit(X_train, y_train)

In [ ]:
# Predict probabilities
proba_hgb = hgb.predict_proba(X_test)[:, 1]

# ROC-AUC
roc_hgb = roc_auc_score(y_test, proba_hgb)

# PR-AUC
pr_hgb = average_precision_score(y_test, proba_hgb)

print("Logistic ROC-AUC:", roc)
print("HGB ROC-AUC:", roc_hgb)

print("\nLogistic PR-AUC:", pr_auc)
print("HGB PR-AUC:", pr_hgb)

In [ ]:
plt.figure(figsize=(6,5))

RocCurveDisplay.from_predictions(y_test, proba_test, name="Logistic")
RocCurveDisplay.from_predictions(y_test, proba_hgb, name="HGB")

plt.title("ROC Curve Comparison")
plt.show()

In [ ]:

# COST OPTIMISATION FOR HGB

thresholds = np.linspace(0.01, 0.99, 99)
costs_hgb = []

for t in thresholds:
    preds = (proba_hgb >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    costs_hgb.append(COST_FN * fn + COST_FP * fp)

best_i_hgb = int(np.argmin(costs_hgb))
best_t_hgb = thresholds[best_i_hgb]

print("Best threshold (HGB):", best_t_hgb)
print("Min expected cost (HGB):", costs_hgb[best_i_hgb])

In [ ]:
best_preds_hgb = (proba_hgb >= best_t_hgb).astype(int)

print("Confusion matrix (HGB @ best_t):\n", confusion_matrix(y_test, best_preds_hgb))
print(classification_report(y_test, best_preds_hgb, digits=3))

In [ ]:
# Plot cost curve comparison (Logistic vs HGB)
plt.plot(thresholds, costs, label="Logistic")
plt.plot(thresholds, costs_hgb, label="HGB")
plt.title("Expected Cost vs Threshold (Model Comparison)")
plt.xlabel("Threshold")
plt.ylabel("Expected Cost")
plt.legend()
plt.show()

HGB is better for cost-sensitive decision making.